In [1]:
import pandas as pd

In [2]:
df=pd.read_excel("encuestas_concatenadas.xlsx")

In [3]:
df

,encuestadora,fecha,factor,departamento,municipio,region,zona,genero,edad_grupo,estrato,...,sv_cepeda_vs_lizcano,sv_cepeda_vs_murillo,sv_cepeda_vs_pinzon,sv_cepeda_vs_uribe_londono,sv_cepeda_vs_valencia,sv_espriella_vs_fajardo,sv_espriella_vs_valencia,sv_fajardo_vs_pinzon,sv_fajardo_vs_valencia,aprobacion_petro
0,Atlas Intel,2026-01-09,2.650916e-06,Santander,Barichara,Central,NaN,Hombre,45-59,NaN,...,NaN,NaN,Voto en blanco,NaN,Iván Cepeda,Sergio Fajardo,NaN,NaN,NaN,Regular
1,Atlas Intel,2026-01-09,6.110878e-07,Antioquia,Medellín,Central,NaN,Hombre,60+,NaN,...,NaN,NaN,Voto en blanco,NaN,Voto en blanco,Voto en blanco,NaN,NaN,NaN,Malo / muy malo
2,Atlas Intel,2026-01-09,7.076182e-07,Antioquia,Medellín,Central,NaN,Hombre,35-44,NaN,...,NaN,NaN,Juan Carlos Pinzón,NaN,Paloma Valencia,Sergio Fajardo,NaN,NaN,NaN,Malo / muy malo
3,Atlas Intel,2026-01-09,7.413255e-07,Antioquia,Medellín,Central,NaN,Mujer,60+,NaN,...,NaN,NaN,Juan Carlos Pinzón,NaN,Voto en blanco,Sergio Fajardo,NaN,NaN,NaN,Malo / muy malo
4,Atlas Intel,2026-01-09,6.080302e-07,Antioquia,Caldas,Central,NaN,Hombre,35-44,NaN,...,NaN,NaN,Voto en blanco,NaN,Voto en blanco,Voto en blanco,NaN,NaN,NaN,Malo / muy malo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44512,CNC,2026-03-16,5.157391e+03,HUILA,Timaná,Centro - Sur - Amazonía,Urbana,Mujer,35-44,2,...,Ninguno,Ninguno,NaN,Miguel Uribe Londoño,Paloma Valencia,NaN,NaN,NaN,NaN,Negativa
44513,CNC,2026-03-16,2.642461e+04,RISARALDA,Pereira,Eje Cafetero,Urbana,Mujer,35-44,3,...,Mauricio Lizcano,Luis Gilberto Murillo,NaN,Miguel Uribe Londoño,Paloma Valencia,NaN,NaN,NaN,NaN,Negativa
44514,CNC,2026-03-16,2.966080e+04,RISARALDA,Pereira,Eje Cafetero,Urbana,Hombre,35-44,3,...,Mauricio Lizcano,Luis Gilberto Murillo,NaN,Miguel Uribe Londoño,Paloma Valencia,NaN,NaN,NaN,NaN,Negativa
44515,CNC,2026-03-16,2.633356e+04,"BOGOTÁ, D.C.","Bogotá, D.C.",Bogotá,Urbana,Hombre,35-44,2,...,Iván Cepeda,Iván Cepeda,NaN,Iván Cepeda,Iván Cepeda,NaN,NaN,NaN,NaN,Positiva


In [4]:
df["encuestadora"]=df["encuestadora"].replace({"Invamer (Colombia Opina 21)": "Invamer","Invamer (Colombia Opina 20)": "Invamer"}, inplace=True)

C:\Users\pablo\AppData\Local\Temp\ipykernel_28772\3794612711.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["encuestadora"]=df["encuestadora"].replace({"Invamer (Colombia Opina 21)": "Invamer","Invamer (Colombia Opina 20)": "Invamer"}, inplace=True)


In [5]:
df["encuestadora"].value_counts()

encuestadora
Atlas Intel    26194
Invamer         7602
CNC             5841
GAD3            4880
Name: count, dtype: int64

In [6]:

# Columnas de voto disponibles en el archivo
PREGUNTAS = [
    'primera_vuelta', 'primera_vuelta_espontanea',
    'sv_cepeda_vs_espriella', 'sv_cepeda_vs_fajardo', 'sv_cepeda_vs_valencia',
    'sv_espriella_vs_fajardo', 'sv_fajardo_vs_valencia', 'sv_espriella_vs_valencia',
    'sv_cepeda_vs_pinzon', 'sv_fajardo_vs_pinzon',
    'sv_cepeda_vs_claudia', 'sv_cepeda_vs_davila', 'sv_cepeda_vs_galan',
    'sv_cepeda_vs_gaviria', 'sv_cardenas_vs_cepeda', 'sv_barreras_vs_cepeda',
    'sv_cepeda_vs_lizcano', 'sv_cepeda_vs_murillo', 'sv_cepeda_vs_uribe_londono',
    'aprobacion_petro',
]
PREGUNTAS = [p for p in PREGUNTAS if p in df.columns]

def pcts(subdf, pregunta):
    validos = subdf[subdf[pregunta].notna()]
    if validos.empty:
        return None
    res = validos.groupby(pregunta)['factor'].sum()
    return (res / res.sum() * 100).round(1).sort_values(ascending=False)

def revisar(encuestadora, fecha=None):
    """Muestra % ponderados por pregunta. Siempre separado por fecha."""
    mask = df['encuestadora'] == encuestadora
    if fecha:
        mask &= df['fecha'].astype(str).str.startswith(str(fecha)[:10])
    sub = df[mask]
    if sub.empty:
        print('Sin datos'); return

    for f, grupo in sub.groupby('fecha'):
        f_str = str(f)[:10]
        print(f"\n{'='*58}\n  {encuestadora}  |  {f_str}  (n={len(grupo):,})\n{'='*58}")
        for preg in PREGUNTAS:
            serie = pcts(grupo, preg)
            if serie is None:
                continue
            print(f"\n  [{preg}]")
            for val, p in serie.items():
                bar = '█' * int(p / 2)
                print(f"    {str(val):<35} {p:>5.1f}%  {bar}")


In [7]:
# Surveys disponibles
df.groupby(['encuestadora','fecha']).size().rename('n').reset_index().sort_values(['encuestadora','fecha'])

,encuestadora,fecha,n
0,Atlas Intel,2026-01-09,4520
1,Atlas Intel,2026-02-05,7298
2,Atlas Intel,2026-02-28,6468
3,Atlas Intel,2026-03-12,4291
4,Atlas Intel,2026-04-10,3617
5,CNC,2026-02-20,3684
6,CNC,2026-03-16,2157
7,GAD3,2026-01-13,1207
8,GAD3,2026-02-16,2473
9,GAD3,2026-03-16,1200


In [8]:
# Atlas Intel — por fecha
revisar('Atlas Intel')


  Atlas Intel  |  2026-01-09  (n=4,520)

  [primera_vuelta]
    Abelardo de la Espriella             28.0%  ██████████████
    Iván Cepeda                          26.5%  █████████████
    Sergio Fajardo                        9.4%  ████
    Voto en blanco                        7.2%  ███
    No sé                                 5.7%  ██
    Juan Carlos Pinzón                    5.1%  ██
    Paloma Valencia                       5.1%  ██
    Claudia López                         2.6%  █
    Enrique Peñalosa                      2.3%  █
    Juan Daniel Oviedo                    1.8%  
    Aníbal Gaviria                        1.3%  
    Juan Manuel Galán                     1.1%  
    No votaría                            1.1%  
    David Luna                            0.9%  
    Vicky Dávila                          0.9%  
    Daniel Quintero                       0.4%  
    Mauricio Cárdenas                     0.4%  
    Roy Barreras                          0.2%  
    Daniel Pala

In [9]:
# GAD3 — por fecha
revisar('GAD3')


  GAD3  |  2026-01-13  (n=1,207)

  [primera_vuelta]
    Iván Cepeda                          30.3%  ███████████████
    Abelardo de la Espriella             21.9%  ██████████
    NS/NR                                14.0%  ███████
    Ninguno                              10.6%  █████
    Voto en blanco                        5.0%  ██
    Paloma Valencia                       3.1%  █
    Juan Manuel Galán                     2.1%  █
    Juan Carlos Pinzón                    1.7%  
    Vicky Dávila                          1.7%  
    Sergio Fajardo                        1.3%  
    Juan Daniel Oviedo                    1.1%  
    Aníbal Gaviria                        1.1%  
    Otro                                  1.0%  
    Santiago Botero                       1.0%  
    María Fernanda Cabal                  0.8%  
    Camilo Romero                         0.7%  
    Roy Barreras                          0.7%  
    Enrique Peñalosa                      0.5%  
    Germán Vargas Llera

In [10]:
# Invamer — Colombia Opina 21
revisar('Invamer')


  Invamer  |  2026-02-25  (n=3,801)

  [primera_vuelta]
    Iván Cepeda                          33.3%  ████████████████
    Ninguno                              19.2%  █████████
    Abelardo de la Espriella             17.0%  ████████
    Claudia López                        11.3%  █████
    Sergio Fajardo                        8.2%  ████
    Paloma Valencia                       7.6%  ███
    Voto en blanco                        1.7%  
    Roy Barreras                          1.5%  
    P245                                  0.0%  

  [primera_vuelta_espontanea]
    Iván Cepeda                          33.4%  ████████████████
    Ninguno                              18.6%  █████████
    Abelardo de la Espriella             17.8%  ████████
    Claudia López                        12.1%  ██████
    Sergio Fajardo                        8.8%  ████
    Paloma Valencia                       6.5%  ███
    Voto en blanco                        1.6%  
    6                                

In [11]:
# CNC — Febrero
revisar('CNC')


  CNC  |  2026-02-20  (n=3,684)

  [primera_vuelta]
    Iván Cepeda                          35.4%  █████████████████
    Abelardo de la Espriella             16.7%  ████████
    NS/NR                                 8.8%  ████
    Ninguno                               6.3%  ███
    Claudia López                         5.2%  ██
    Sergio Fajardo                        4.8%  ██
    Paloma Valencia                       4.1%  ██
    Juan Manuel Galán                     2.6%  █
    Vicky Dávila                          2.0%  █
    Juan Daniel Oviedo                    1.8%  
    Santiago Botero                       1.7%  
    Daniel Quintero                       1.5%  
    Voto en blanco                        1.2%  
    Enrique Peñalosa                      1.2%  
    Carlos Caicedo                        1.2%  
    Miguel Uribe Londoño                  1.0%  
    Juan Carlos Pinzón                    1.0%  
    Roy Barreras                          0.9%  
    Aníbal Gaviria       

In [12]:
# CNC — Marzo
revisar('CNC', '2026-03-16')


  CNC  |  2026-03-16  (n=2,157)

  [primera_vuelta]
    Iván Cepeda                          34.5%  █████████████████
    Paloma Valencia                      22.2%  ███████████
    Abelardo de la Espriella             15.5%  ███████
    NS/NR                                 8.0%  ████
    Voto en blanco                        6.5%  ███
    Claudia López                         3.7%  █
    Sergio Fajardo                        3.6%  █
    Ninguno                               1.9%  
    Santiago Botero                       1.3%  
    Miguel Uribe Londoño                  1.0%  
    Roy Barreras                          0.5%  
    Sondra Macollins                      0.4%  
    Clara López                           0.3%  
    Gustavo Matamoros                     0.2%  
    Luis Gilberto Murillo                 0.2%  
    Mauricio Lizcano                      0.2%  
    Carlos Caicedo                        0.1%  

  [sv_cepeda_vs_espriella]
    Iván Cepeda                          4

## Trasbase — transferencia de votos primera vuelta → segunda vuelta

In [13]:

SV_COLS = [c for c in df.columns if c.startswith('sv_')]

def trasbase(encuestadora, fecha=None):
    """
    Para cada escenario de segunda vuelta muestra, entre quienes votan por
    cada candidato en primera vuelta, cómo se distribuye el voto (ponderado).
    """
    mask = df['encuestadora'] == encuestadora
    if fecha:
        mask &= df['fecha'].astype(str).str.startswith(str(fecha)[:10])
    sub = df[mask]

    for f, grupo in sub.groupby('fecha'):
        f_str = str(f)[:10]
        print(f"\n{'='*65}")
        print(f"  TRASBASE  |  {encuestadora}  |  {f_str}  (n={len(grupo):,})")
        print(f"{'='*65}")

        for sv_col in SV_COLS:
            validos = grupo[grupo[sv_col].notna() & grupo['primera_vuelta'].notna()].copy()
            if validos.empty:
                continue

            # Candidatos del escenario sv (excluir NS/NR, blancos, etc.)
            opciones = validos[sv_col].value_counts()
            candidatos_sv = [v for v in opciones.index]
            if len(candidatos_sv) < 2:
                continue

            print(f"\n  [{sv_col}]")

            # Tabla: filas = primera_vuelta, columnas = voto en sv
            ct = (validos
                  .groupby(['primera_vuelta', sv_col])['factor']
                  .sum()
                  .unstack(fill_value=0))

            # Ordenar columnas: candidatos primero, luego el resto
            col_order = candidatos_sv + [c for c in ct.columns if c not in candidatos_sv]
            ct = ct[[c for c in col_order if c in ct.columns]]

            # Normalizar por fila (% dentro de cada candidato de primera vuelta)
            ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

            # Ordenar filas por peso total descendente
            totales = validos.groupby('primera_vuelta')['factor'].sum().sort_values(ascending=False)
            filas_orden = [f for f in totales.index if f in ct_pct.index]
            ct_pct = ct_pct.loc[filas_orden]

            # Header
            col_labels = [str(c)[:18] for c in ct_pct.columns]
            header = f"    {'Primera vuelta':<28}" + "".join(f"{l:>10}" for l in col_labels)
            print(header)
            print("    " + "-" * (28 + 10 * len(col_labels)))

            for candidato_1v, row in ct_pct.iterrows():
                n_pond = int(totales.get(candidato_1v, 0))
                label = str(candidato_1v)[:26]
                vals = "".join(f"{v:>9.1f}%" for v in row.values)
                print(f"    {label:<28}{vals}  (n~{n_pond:,})")


In [14]:
trasbase('Atlas Intel')


  TRASBASE  |  Atlas Intel  |  2026-01-09  (n=4,520)

  [sv_cepeda_vs_espriella]
    Primera vuelta              Iván CepedaAbelardo de la EspVoto en blanco     No sé
    --------------------------------------------------------------------
    Abelardo de la Espriella          0.2%     98.5%      1.1%      0.2%  (n~0)
    Iván Cepeda                      98.1%      0.4%      1.4%      0.2%  (n~0)
    Sergio Fajardo                   34.3%     29.2%     21.8%     14.7%  (n~0)
    Voto en blanco                    7.8%      4.1%     45.5%     42.6%  (n~0)
    No sé                             4.4%     11.3%     10.1%     74.2%  (n~0)
    Juan Carlos Pinzón               25.1%     63.6%     10.9%      0.5%  (n~0)
    Paloma Valencia                  16.3%     76.6%      7.0%      0.0%  (n~0)
    Claudia López                    27.1%     39.3%     28.1%      5.4%  (n~0)
    Enrique Peñalosa                 34.6%     29.5%      0.3%     35.5%  (n~0)
    Juan Daniel Oviedo               17

In [15]:
trasbase('GAD3')


  TRASBASE  |  GAD3  |  2026-01-13  (n=1,207)

  [sv_cepeda_vs_espriella]
    Primera vuelta              Iván CepedaAbelardo de la EspVoto en blancoNo votaría     NS/NR
    ------------------------------------------------------------------------------
    Iván Cepeda                      98.3%      0.8%      0.5%      0.4%      0.0%  (n~365)
    Abelardo de la Espriella          1.4%     98.6%      0.0%      0.0%      0.0%  (n~264)
    NS/NR                            14.8%     17.3%     18.9%      6.9%     42.2%  (n~169)
    Ninguno                          14.2%      5.0%     14.7%     58.0%      8.1%  (n~128)
    Voto en blanco                   25.5%      4.9%     56.3%      9.4%      3.7%  (n~59)
    Paloma Valencia                  16.1%     80.4%      0.0%      3.6%      0.0%  (n~37)
    Juan Manuel Galán                19.6%     23.9%     19.6%     23.9%     12.9%  (n~25)
    Vicky Dávila                     44.0%     28.0%     28.1%      0.0%      0.0%  (n~21)
    Juan Carlo

In [16]:
trasbase('Invamer')



  TRASBASE  |  Invamer  |  2026-02-25  (n=3,801)

  [sv_cepeda_vs_espriella]
    Primera vuelta              Iván CepedaAbelardo de la Esp   NingunoVoto en blanco      P233
    ------------------------------------------------------------------------------
    Iván Cepeda                      98.1%      1.2%      0.6%      0.1%      0.0%  (n~1,266)
    Ninguno                           9.9%      6.9%     82.2%      1.0%      0.0%  (n~730)
    Abelardo de la Espriella          1.0%     98.2%      0.8%      0.0%      0.0%  (n~646)
    Claudia López                    43.3%     25.7%     27.2%      3.9%      0.0%  (n~429)
    Sergio Fajardo                   40.2%     27.5%     24.7%      7.6%      0.0%  (n~313)
    Paloma Valencia                  25.7%     55.4%     16.6%      2.3%      0.0%  (n~289)
    Voto en blanco                   12.2%      4.1%     20.2%     63.5%      0.0%  (n~66)
    Roy Barreras                     49.4%     29.9%     18.9%      1.8%      0.0%  (n~58)
    P24

In [17]:
trasbase('CNC')


  TRASBASE  |  CNC  |  2026-02-20  (n=3,684)

  [sv_cardenas_vs_cepeda]
    Primera vuelta              Iván Cepeda   NingunoMauricio Cárdenas     NS/NR
    --------------------------------------------------------------------
    Iván Cepeda                      99.0%      0.3%      0.5%      0.2%  (n~14,062,716)
    Abelardo de la Espriella          8.9%     49.6%     38.8%      2.6%  (n~6,641,961)
    NS/NR                            42.9%     15.8%      6.3%     34.9%  (n~3,489,434)
    Ninguno                          32.8%     56.2%      4.2%      6.8%  (n~2,486,649)
    Claudia López                    69.4%     14.3%     12.4%      3.9%  (n~2,055,982)
    Sergio Fajardo                   40.0%     35.6%     21.7%      2.7%  (n~1,914,474)
    Paloma Valencia                  21.2%     34.2%     42.4%      2.3%  (n~1,636,089)
    Juan Manuel Galán                46.1%     29.1%     20.4%      4.3%  (n~1,013,735)
    Vicky Dávila                     44.3%     21.7%     29.1%      

## ID resultado por encuesta

In [18]:

# Tabla de IDs: (encuestadora normalizada, fecha YYYY-MM-DD) → id_resultado
ID_MAP = {
    ("Atlas Intel",  "2026-01-09"): "574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5",
    ("Atlas Intel",  "2026-02-05"): "dd8c6d8f-8414-46cc-a4a3-0d4c0cb5d727",
    ("Atlas Intel",  "2026-02-28"): "8f1dc697-304b-41ca-a83c-bfa5e8df23c9",
    ("Atlas Intel",  "2026-03-12"): "8e690c54-bb74-4905-8432-c1b5d5ba995a",
    ("Atlas Intel",  "2026-04-10"): "ab056bb2-8345-49d7-a4bf-6428896acdf8",
    ("CNC",          "2026-02-20"): "f8780e6c-447a-4345-9934-9b711f849120",
    ("CNC",          "2026-03-16"): "4d6479aa-58b2-4399-924c-03ffb307054d",
    ("GAD3",         "2026-01-13"): "ccd56d3f-a8e8-4de4-a451-134c43038029",
    ("GAD3",         "2026-02-16"): "1197a175-79ed-48e8-9822-ced8ec0998bb",
    ("GAD3",         "2026-03-16"): "762692c5-2f67-4b91-8fdf-93b662e7f7e7",
    ("Invamer",      "2026-02-25"): "1fb623ba-a1f0-42b3-8c08-9580d41eca23",
    ("Invamer",      "2026-04-25"): "a0c3e256-a5b3-4157-bae4-eb5ff479fb7f",
}

def _enc_key(enc):
    """Normaliza el nombre de la encuestadora al prefijo usado en ID_MAP."""
    if enc.startswith("Invamer"):
        return "Invamer"
    return enc

def get_id(encuestadora, fecha):
    """Devuelve el id_resultado dado encuestadora y fecha (str o Timestamp)."""
    key = (_enc_key(encuestadora), str(fecha)[:10])
    return ID_MAP.get(key)

# Agregar columna id_resultado al df
df["id_resultado"] = df.apply(lambda row: get_id(row["encuestadora"], row["fecha"]), axis=1)

# Verificar cobertura
print("Cobertura de id_resultado:")
print(df.groupby(["encuestadora", "fecha"])["id_resultado"].first().to_string())
print(f"\nFilas sin ID: {df['id_resultado'].isna().sum()}")


Cobertura de id_resultado:
encuestadora  fecha     
Atlas Intel   2026-01-09    574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5
              2026-02-05    dd8c6d8f-8414-46cc-a4a3-0d4c0cb5d727
              2026-02-28    8f1dc697-304b-41ca-a83c-bfa5e8df23c9
              2026-03-12    8e690c54-bb74-4905-8432-c1b5d5ba995a
              2026-04-10    ab056bb2-8345-49d7-a4bf-6428896acdf8
CNC           2026-02-20    f8780e6c-447a-4345-9934-9b711f849120
              2026-03-16    4d6479aa-58b2-4399-924c-03ffb307054d
GAD3          2026-01-13    ccd56d3f-a8e8-4de4-a451-134c43038029
              2026-02-16    1197a175-79ed-48e8-9822-ced8ec0998bb
              2026-03-16    762692c5-2f67-4b91-8fdf-93b662e7f7e7
Invamer       2026-02-25    1fb623ba-a1f0-42b3-8c08-9580d41eca23
              2026-04-25    a0c3e256-a5b3-4157-bae4-eb5ff479fb7f

Filas sin ID: 0


In [19]:
df.columns

Index(['encuestadora', 'fecha', 'factor', 'departamento', 'municipio',
       'region', 'zona', 'genero', 'edad_grupo', 'estrato', 'educacion',
       'primera_vuelta', 'primera_vuelta_espontanea', 'sv_barreras_vs_cepeda',
       'sv_cardenas_vs_cepeda', 'sv_cepeda_vs_claudia', 'sv_cepeda_vs_davila',
       'sv_cepeda_vs_espriella', 'sv_cepeda_vs_fajardo', 'sv_cepeda_vs_galan',
       'sv_cepeda_vs_gaviria', 'sv_cepeda_vs_lizcano', 'sv_cepeda_vs_murillo',
       'sv_cepeda_vs_pinzon', 'sv_cepeda_vs_uribe_londono',
       'sv_cepeda_vs_valencia', 'sv_espriella_vs_fajardo',
       'sv_espriella_vs_valencia', 'sv_fajardo_vs_pinzon',
       'sv_fajardo_vs_valencia', 'aprobacion_petro', 'id_resultado'],
      dtype='str')

In [20]:
df[df["encuestadora"]=="CNC"]["fecha"].value_counts()

fecha
2026-02-20    3684
2026-03-16    2157
Name: count, dtype: int64

In [21]:
res = (
    df[(df["encuestadora"] == "CNC") & (df["fecha"] == "2026-03-16")]
    .groupby("primera_vuelta")["factor"]
    .sum()
)

res_norm = res / res.sum()

In [22]:
res_norm

primera_vuelta
Abelardo de la Espriella    0.154573
Carlos Caicedo              0.001248
Clara López                 0.003005
Claudia López               0.036998
Gustavo Matamoros           0.002397
Iván Cepeda                 0.344733
Luis Gilberto Murillo       0.002191
Mauricio Lizcano            0.001859
Miguel Uribe Londoño        0.010283
NS/NR                       0.079652
Ninguno                     0.018580
Paloma Valencia             0.221839
Roy Barreras                0.004786
Santiago Botero             0.012690
Sergio Fajardo              0.035929
Sondra Macollins            0.003876
Voto en blanco              0.065361
Name: factor, dtype: float64

In [23]:

# ── Versión generalizada: cruza cualquier par de columnas ──────────────────

def build_crosstab_long(df, col_fila, col_columna, escenarios=None):
    """
    Cruza col_fila × col_columna con % ponderados, en formato long.

    Sin escenarios: cruza las dos columnas directamente.
    Con escenarios (dict {nombre: col_real_en_df}): itera sobre cada
      escenario y añade columna 'escenario'; col_columna es el nombre de salida.

    Columnas resultado:
        encuestadora, fecha, [escenario,] <col_fila>, <col_columna>, pct
    """
    def _pair(sub, fila, col):
        partes = []
        for (enc, fecha), grupo in sub.groupby(['encuestadora', 'fecha']):
            ct = grupo.groupby([fila, col])['factor'].sum().reset_index()
            tots = ct.groupby(fila)['factor'].sum()
            ct['_total_fila'] = ct[fila].map(tots)
            ct = ct[ct['_total_fila'] > 0]
            if ct.empty:
                continue
            ct['pct'] = (ct['factor'] / ct['_total_fila'] * 100).round(1)
            ct = ct.drop(columns=['_total_fila'])
            ct['encuestadora'] = enc
            ct['fecha'] = fecha
            partes.append(ct[['encuestadora', 'fecha', fila, col, 'pct']])
        return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

    if escenarios:
        bloques = []
        for nombre, col_real in escenarios.items():
            if col_real not in df.columns:
                continue
            sub = df[df[col_fila].notna() & df[col_real].notna()]
            bloque = _pair(sub, col_fila, col_real)
            if bloque.empty:
                continue
            bloque = bloque.rename(columns={col_real: col_columna})
            bloque.insert(2, 'escenario', nombre)
            bloques.append(bloque)
        return pd.concat(bloques, ignore_index=True) if bloques else pd.DataFrame()

    sub = df[df[col_fila].notna() & df[col_columna].notna()]
    return _pair(sub, col_fila, col_columna)


# Los 6 escenarios clave (reemplaza el ESCENARIOS anterior con los 6 completos)
ESCENARIOS_6 = {
    'cepeda_vs_paloma':    'sv_cepeda_vs_valencia',
    'cepeda_vs_abelardo':  'sv_cepeda_vs_espriella',
    'cepeda_vs_fajardo':   'sv_cepeda_vs_fajardo',
    'paloma_vs_fajardo':   'sv_fajardo_vs_valencia',
    'abelardo_vs_fajardo': 'sv_espriella_vs_fajardo',
    'paloma_vs_abelardo':  'sv_espriella_vs_valencia',
}

# Uso: replicar trasbase de los 6 escenarios
df_long6 = build_crosstab_long(df, 'primera_vuelta', 'voto_sv', escenarios=ESCENARIOS_6)
print(f"Filas: {len(df_long6):,}  |  Escenarios: {df_long6['escenario'].unique().tolist()}")
df_long6.head(10)


Filas: 3,282  |  Escenarios: ['cepeda_vs_paloma', 'cepeda_vs_abelardo', 'cepeda_vs_fajardo', 'paloma_vs_fajardo', 'abelardo_vs_fajardo', 'paloma_vs_abelardo']


,encuestadora,fecha,escenario,primera_vuelta,voto_sv,pct
0,Atlas Intel,2026-01-09,cepeda_vs_paloma,Abelardo de la Espriella,Iván Cepeda,5.5
1,Atlas Intel,2026-01-09,cepeda_vs_paloma,Abelardo de la Espriella,No sé,4.3
2,Atlas Intel,2026-01-09,cepeda_vs_paloma,Abelardo de la Espriella,Paloma Valencia,71.7
3,Atlas Intel,2026-01-09,cepeda_vs_paloma,Abelardo de la Espriella,Voto en blanco,18.5
4,Atlas Intel,2026-01-09,cepeda_vs_paloma,Aníbal Gaviria,Iván Cepeda,31.6
5,Atlas Intel,2026-01-09,cepeda_vs_paloma,Aníbal Gaviria,No sé,36.2
6,Atlas Intel,2026-01-09,cepeda_vs_paloma,Aníbal Gaviria,Paloma Valencia,27.9
7,Atlas Intel,2026-01-09,cepeda_vs_paloma,Aníbal Gaviria,Voto en blanco,4.3
8,Atlas Intel,2026-01-09,cepeda_vs_paloma,Claudia López,Iván Cepeda,23.5
9,Atlas Intel,2026-01-09,cepeda_vs_paloma,Claudia López,No sé,5.5


In [24]:

# ── Ponderación entre encuestas ────────────────────────────────────────────

# w_total por (encuestadora, fecha_pub) — tabla de pesos suministrada
PESOS = {
    ('Invamer',     '2026-04-25'): 1766,
    ('Atlas Intel', '2026-04-10'):  707,
    ('GAD3',        '2026-03-16'):   89,
    ('CNC',         '2026-03-16'):  220,
    ('Atlas Intel', '2026-03-12'):  175,
    ('Atlas Intel', '2026-02-28'):   29,
    ('Invamer',     '2026-02-25'):   22,
    ('CNC',         '2026-02-20'):   26,
    ('GAD3',        '2026-02-16'):   15,
    ('Atlas Intel', '2026-02-05'):   29,
    ('GAD3',        '2026-01-13'):    1,
    ('Atlas Intel', '2026-01-09'):  175,
}


def ponderar(df_vals, col_valor='pct', groupby=None,
             col_enc='encuestadora', col_fecha='fecha'):
    """
    Promedio ponderado de col_valor usando PESOS.

    Sin groupby  → escalar (promedio global).
    Con groupby  → DataFrame con una fila por grupo y columna '<col_valor>_pond'.

    Solo se usan filas cuyo (encuestadora, fecha) tenga peso en PESOS.
    El promedio es:  sum(valor * w) / sum(w)
    """
    d = df_vals.copy()
    d['_w'] = d.apply(
        lambda r: PESOS.get((r[col_enc], str(r[col_fecha])[:10]), 0), axis=1
    )
    d = d[d['_w'] > 0]
    if d.empty:
        return None

    if groupby:
        def _wavg(g):
            tw = g['_w'].sum()
            return round((g[col_valor] * g['_w']).sum() / tw, 2) if tw else None
        return (
            d.groupby(groupby)
             .apply(_wavg, include_groups=False)
             .reset_index(name=f'{col_valor}_pond')
        )

    tw = d['_w'].sum()
    return round((d[col_valor] * d['_w']).sum() / tw, 2)

def aplicar_peso_encuesta(df_vals, col_enc='encuestadora', col_fecha='fecha'):
    d = df_vals.copy()
    d['_w_enc'] = d.apply(lambda r: PESOS.get((r[col_enc], str(r[col_fecha])[:10]), 0), axis=1)
    return d[d['_w_enc'] > 0].copy()

def crosstab_ponderado(df_vals, fila, columna, usar_pesos_encuesta=False, col_enc='encuestadora', col_fecha='fecha'):
    sub = df_vals[df_vals[fila].notna() & df_vals[columna].notna()].copy()
    if sub.empty:
        return pd.DataFrame()
    if usar_pesos_encuesta:
        sub = aplicar_peso_encuesta(sub, col_enc=col_enc, col_fecha=col_fecha)
        if sub.empty:
            return pd.DataFrame()
        peso_col = '_w_enc'
        sub['_peso_total'] = sub['factor'] * sub[peso_col]
        peso_col = '_peso_total'
    else:
        peso_col = 'factor'
    ct = sub.groupby([fila, columna])[peso_col].sum().unstack(fill_value=0)
    total_fila = ct.sum(axis=1).replace(0, pd.NA)
    return (ct.div(total_fila, axis=0) * 100).round(1).fillna(0)

def sesgo_por_candidato_df(subdf, col_candidato='primera_vuelta', col_demo='edad_grupo'):
    validos = subdf[subdf[col_candidato].notna() & subdf[col_demo].notna()].copy()
    if validos.empty:
        return pd.DataFrame()
    ct_pond = validos.pivot_table(index=col_candidato, columns=col_demo, values='factor', aggfunc='sum', fill_value=0)
    ct_raw = validos.pivot_table(index=col_candidato, columns=col_demo, values='factor', aggfunc='size', fill_value=0)
    pond_total = ct_pond.sum(axis=1).replace(0, pd.NA)
    raw_total = ct_raw.sum(axis=1).replace(0, pd.NA)
    ct_pond = (ct_pond.div(pond_total, axis=0) * 100).round(1).fillna(0)
    ct_raw = (ct_raw.div(raw_total, axis=0) * 100).round(1).fillna(0)
    pond = ct_pond.reset_index().melt(id_vars=col_candidato, var_name=col_demo, value_name='ponderado_pct')
    raw = ct_raw.reset_index().melt(id_vars=col_candidato, var_name=col_demo, value_name='sin_peso_pct')
    tabla = pond.merge(raw, on=[col_candidato, col_demo], how='outer').fillna(0)
    tabla['sesgo_pp'] = (tabla['ponderado_pct'] - tabla['sin_peso_pct']).round(1)
    return tabla.sort_values([col_candidato, 'sesgo_pp'], ascending=[True, False])


# Ejemplo de uso: ponderar(df_long6, groupby=['escenario', 'primera_vuelta', 'voto_sv'])


In [25]:
df[["edad_grupo","encuestadora"]].value_counts()

edad_grupo  encuestadora
45-59       Atlas Intel     7459
35-44       Atlas Intel     5904
25-34       Atlas Intel     5224
60+         Atlas Intel     5202
55+         Invamer         2547
18-24       Atlas Intel     2405
55+         CNC             1753
            GAD3            1670
25-34       Invamer         1474
35-44       Invamer         1358
45-54       Invamer         1239
25-34       CNC             1188
35-44       CNC             1084
45-54       CNC             1073
18-24       Invamer          982
35-44       GAD3             801
45-54       GAD3             793
18-24       CNC              743
25-34       GAD3             628
35-54       GAD3             485
18-34       GAD3             295
18-24       GAD3             208
Name: count, dtype: int64

In [26]:
df

,encuestadora,fecha,factor,departamento,municipio,region,zona,genero,edad_grupo,estrato,...,sv_cepeda_vs_murillo,sv_cepeda_vs_pinzon,sv_cepeda_vs_uribe_londono,sv_cepeda_vs_valencia,sv_espriella_vs_fajardo,sv_espriella_vs_valencia,sv_fajardo_vs_pinzon,sv_fajardo_vs_valencia,aprobacion_petro,id_resultado
0,Atlas Intel,2026-01-09,2.650916e-06,Santander,Barichara,Central,NaN,Hombre,45-59,NaN,...,NaN,Voto en blanco,NaN,Iván Cepeda,Sergio Fajardo,NaN,NaN,NaN,Regular,574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5
1,Atlas Intel,2026-01-09,6.110878e-07,Antioquia,Medellín,Central,NaN,Hombre,60+,NaN,...,NaN,Voto en blanco,NaN,Voto en blanco,Voto en blanco,NaN,NaN,NaN,Malo / muy malo,574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5
2,Atlas Intel,2026-01-09,7.076182e-07,Antioquia,Medellín,Central,NaN,Hombre,35-44,NaN,...,NaN,Juan Carlos Pinzón,NaN,Paloma Valencia,Sergio Fajardo,NaN,NaN,NaN,Malo / muy malo,574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5
3,Atlas Intel,2026-01-09,7.413255e-07,Antioquia,Medellín,Central,NaN,Mujer,60+,NaN,...,NaN,Juan Carlos Pinzón,NaN,Voto en blanco,Sergio Fajardo,NaN,NaN,NaN,Malo / muy malo,574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5
4,Atlas Intel,2026-01-09,6.080302e-07,Antioquia,Caldas,Central,NaN,Hombre,35-44,NaN,...,NaN,Voto en blanco,NaN,Voto en blanco,Voto en blanco,NaN,NaN,NaN,Malo / muy malo,574bebb8-d5bc-4a9a-b0fc-5a5471c2a8d5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44512,CNC,2026-03-16,5.157391e+03,HUILA,Timaná,Centro - Sur - Amazonía,Urbana,Mujer,35-44,2,...,Ninguno,NaN,Miguel Uribe Londoño,Paloma Valencia,NaN,NaN,NaN,NaN,Negativa,4d6479aa-58b2-4399-924c-03ffb307054d
44513,CNC,2026-03-16,2.642461e+04,RISARALDA,Pereira,Eje Cafetero,Urbana,Mujer,35-44,3,...,Luis Gilberto Murillo,NaN,Miguel Uribe Londoño,Paloma Valencia,NaN,NaN,NaN,NaN,Negativa,4d6479aa-58b2-4399-924c-03ffb307054d
44514,CNC,2026-03-16,2.966080e+04,RISARALDA,Pereira,Eje Cafetero,Urbana,Hombre,35-44,3,...,Luis Gilberto Murillo,NaN,Miguel Uribe Londoño,Paloma Valencia,NaN,NaN,NaN,NaN,Negativa,4d6479aa-58b2-4399-924c-03ffb307054d
44515,CNC,2026-03-16,2.633356e+04,"BOGOTÁ, D.C.","Bogotá, D.C.",Bogotá,Urbana,Hombre,35-44,2,...,Iván Cepeda,NaN,Iván Cepeda,Iván Cepeda,NaN,NaN,NaN,NaN,Positiva,4d6479aa-58b2-4399-924c-03ffb307054d


In [27]:
df[["encuestadora","region"]].value_counts()

encuestadora  region                 
Atlas Intel   Central                    10308
              Bogotá                      6581
              Caribe                      4663
              Pacífico                    3750
Invamer       Centro - Oriente            2448
              Caribe                      1728
              Eje Cafetero                1376
              Pacífico                    1216
CNC           Caribe                      1160
              Centro - Oriente             990
              Pacífico                     989
              Eje Cafetero                 969
Atlas Intel   Amazonía - Orinoquía         892
CNC           Bogotá                       830
GAD3          Central                      598
              Caribe                       543
CNC           Centro - Sur - Amazonía      524
Invamer       Centro - Sur - Amazonía      512
GAD3          Centro - Oriente             455
              Pacífico                     400
CNC           Llano   

## Análisis demográfico y sesgos

Sección para responder con código a: sesgos por encuestadora, perfil demográfico de indecisos, distribución por aprobación a Petro y comparación por edad entre primera y segunda vuelta.

In [ ]:
import pandas as pd

if 'df' not in globals():
    df = pd.read_excel("encuestas_concatenadas.xlsx")

# -----------------------------------------------------------------------------
# Herramientas para responder las preguntas demográficas del cuaderno
# -----------------------------------------------------------------------------

DEMOGRAFICAS_CANDIDATAS = [
    'sexo',
    'edad_grupo',
    'region',
    'estrato',
    'nivel_educativo',
    'educacion',
    'nivel_socioeconomico',
    'zona',
    'departamento',
    'municipio',
]

DEMOGRAFICAS = [c for c in DEMOGRAFICAS_CANDIDATAS if c in df.columns]

INDECISO_PATTERNS = (
    'no sabe',
    'no responde',
    'no votaria',
    'no votaría',
    'blanco',
    'ns/nr',
    'nsnr',
    'indeciso',
)

EDAD_EQUIV = {
    '18-24': '18-34',
    '25-34': '18-34',
    '18-34': '18-34',
    '35-44': '35-54',
    '45-54': '35-54',
    '35-54': '35-54',
    '45-59': '55+',
    '55+': '55+',
    '60+': '55+',
}

def _norm_text(value):
    if pd.isna(value):
        return ''
    return str(value).strip().lower()

def _homogeneizar_columna(subdf, col_demo):
    col_cmp = col_demo
    out = subdf.copy()
    if col_demo == 'edad_grupo':
        col_cmp = 'edad_grupo_cmp'
        out[col_cmp] = out[col_demo].astype(str).str.strip().map(EDAD_EQUIV).fillna(out[col_demo].astype(str).str.strip())
        out.loc[out[col_demo].isna(), col_cmp] = pd.NA
    return out, col_cmp

def es_indeciso(value):
    texto = _norm_text(value)
    return any(pat in texto for pat in INDECISO_PATTERNS)

def filtrar_encuesta(encuestadora=None, fecha=None):
    sub = df.copy()
    if encuestadora is not None:
        sub = sub[sub['encuestadora'] == encuestadora]
    if fecha is not None:
        sub = sub[sub['fecha'].astype(str).str.startswith(str(fecha)[:10])]
    return sub

def _weighted_pct(subdf, col):
    validos = subdf[subdf[col].notna()]
    if validos.empty:
        return pd.Series(dtype=float)
    pesos = validos.groupby(col)['factor'].sum().sort_values(ascending=False)
    return (pesos / pesos.sum() * 100).round(1)

def _homogeneizar_edad_en_df(subdf):
    out = subdf.copy()
    if 'edad_grupo' in out.columns:
        out['edad_grupo'] = out['edad_grupo'].astype(str).str.strip().map(EDAD_EQUIV).fillna(out['edad_grupo'].astype(str).str.strip())
        out.loc[out['edad_grupo'].isin(['nan', 'None', '']), 'edad_grupo'] = pd.NA
    return out

def _distribucion_por_encuesta(subdf, col_demo):
    sub2, col_cmp = _homogeneizar_columna(subdf, col_demo)
    sub2 = sub2[sub2[col_cmp].notna()].copy()
    if sub2.empty:
        return pd.DataFrame(), col_cmp

    piezas = []
    for (enc, f), grupo in sub2.groupby(['encuestadora', 'fecha']):
        dist = _weighted_pct(grupo, col_cmp)
        if dist.empty:
            continue
        tmp = dist.rename('pct').reset_index()
        tmp['encuestadora'] = enc
        tmp['fecha'] = f
        piezas.append(tmp[[col_cmp, 'encuestadora', 'fecha', 'pct']])

    if not piezas:
        return pd.DataFrame(), col_cmp
    return pd.concat(piezas, ignore_index=True), col_cmp

def _combinar_distribucion_ponderada(subdf, col_demo):
    base, col_cmp = _distribucion_por_encuesta(subdf, col_demo)
    if base.empty:
        return pd.Series(dtype=float), col_cmp
    combinado = ponderar(base, col_valor='pct', groupby=[col_cmp])
    if combinado is None or combinado.empty:
        return pd.Series(dtype=float), col_cmp
    serie = combinado.set_index(col_cmp)['pct_pond']
    total = serie.sum()
    if total > 0:
        serie = (serie / total * 100).round(1)
    return serie.sort_values(ascending=False), col_cmp

def sesgo_demografico(encuestadora, col_demo, fecha=None):
    sub = filtrar_encuesta(encuestadora=encuestadora, fecha=fecha)
    if col_demo not in sub.columns:
        return pd.DataFrame()
    ponderado, col_cmp = _combinar_distribucion_ponderada(sub, col_demo)
    sub2, col_cmp = _homogeneizar_columna(sub, col_demo)
    sin_peso = (sub2[col_cmp].value_counts(normalize=True) * 100).round(1).rename('sin_peso_pct')
    ponderado = ponderado.rename('ponderado_pct')
    tabla = pd.concat([sin_peso, ponderado], axis=1).fillna(0)
    tabla['sesgo_pp'] = (tabla['ponderado_pct'] - tabla['sin_peso_pct']).round(1)
    return tabla.sort_values('sesgo_pp', ascending=False)

def resumen_sesgos(encuestadora, top_n=5):
    resultados = {}
    for col_demo in DEMOGRAFICAS:
        tabla = sesgo_demografico(encuestadora, col_demo)
        if tabla.empty:
            continue
        resultados[col_demo] = {
            'sobrepondera': tabla['sesgo_pp'].head(top_n),
            'subpondera': tabla['sesgo_pp'].tail(top_n),
        }
    return resultados

def sesgo_relativo_vs_otras(col_demo, fecha=None):
    sub = filtrar_encuesta(fecha=fecha)
    if col_demo not in sub.columns:
        return pd.DataFrame()
    validos = sub[sub[col_demo].notna() & sub['encuestadora'].notna()].copy()
    if validos.empty:
        return pd.DataFrame()

    filas = []
    encuestadoras = sorted(validos['encuestadora'].unique())

    for enc in encuestadoras:
        sub_enc = validos[validos['encuestadora'] == enc]
        if sub_enc.empty:
            continue
        dist_enc, _ = _combinar_distribucion_ponderada(sub_enc, col_demo)
        if dist_enc.empty:
            continue

        otras = validos[validos['encuestadora'] != enc]
        dist_otras, _ = _combinar_distribucion_ponderada(otras, col_demo)
        if dist_otras.empty:
            continue

        categorias = dist_enc.index.intersection(dist_otras.index)
        if len(categorias) == 0:
            continue

        for cat in categorias:
            peso_enc = round(float(dist_enc.get(cat, 0.0)), 1)
            peso_otras = round(float(dist_otras.get(cat, 0.0)), 1)
            filas.append({
                'encuestadora': enc,
                'variable': col_demo,
                'categoria': cat,
                'peso_encuestadora': peso_enc,
                'peso_promedio_otras': peso_otras,
                'sesgo_rel_pp': round(peso_enc - peso_otras, 1),
            })

    if not filas:
        return pd.DataFrame()

    return pd.DataFrame(filas).sort_values(['variable', 'encuestadora', 'sesgo_rel_pp'], ascending=[True, True, False])

def tabla_sesgos_relativos(variables=None, fecha=None):
    if variables is None:
        variables = DEMOGRAFICAS + (['aprobacion_petro'] if 'aprobacion_petro' in df.columns else [])
    piezas = []
    for var in variables:
        t = sesgo_relativo_vs_otras(var, fecha=fecha)
        if not t.empty:
            piezas.append(t)
    if not piezas:
        return pd.DataFrame()
    return pd.concat(piezas, ignore_index=True)

def perfil_indecisos(col_demo='edad_grupo', fecha=None):
    sub = filtrar_encuesta(fecha=fecha)
    if col_demo not in sub.columns:
        return pd.DataFrame()
    indecisos = sub[sub['primera_vuelta'].apply(es_indeciso)].copy()
    if indecisos.empty:
        return pd.DataFrame()

    serie, _ = _combinar_distribucion_ponderada(indecisos, col_demo)
    if serie.empty:
        return pd.DataFrame()
    combinado = serie.rename('pct_final').to_frame()
    combinado.index.name = col_demo
    return combinado.sort_values('pct_final', ascending=False)

def votos_segun_aprobacion_petro(fecha=None, usar_pesos_encuesta=True):
    sub = filtrar_encuesta(fecha=fecha)
    if 'aprobacion_petro' not in sub.columns:
        return pd.DataFrame()
    return crosstab_ponderado(sub, 'aprobacion_petro', 'primera_vuelta', usar_pesos_encuesta=usar_pesos_encuesta)

def votos_por_edad(col_voto, fecha=None, usar_pesos_encuesta=True):
    sub = filtrar_encuesta(fecha=fecha)
    if 'edad_grupo' not in sub.columns or col_voto not in sub.columns:
        return pd.DataFrame()
    sub = _homogeneizar_edad_en_df(sub)
    return crosstab_ponderado(sub, 'edad_grupo', col_voto, usar_pesos_encuesta=usar_pesos_encuesta)
def sesgo_por_candidato(encuestadora, col_candidato='primera_vuelta', col_demo='edad_grupo', fecha=None):
    sub = filtrar_encuesta(encuestadora=encuestadora, fecha=fecha)
    if col_candidato not in sub.columns or col_demo not in sub.columns:
        return pd.DataFrame()
    if col_demo == 'edad_grupo':
        sub = _homogeneizar_edad_en_df(sub)
    return sesgo_por_candidato_df(sub, col_candidato=col_candidato, col_demo=col_demo)

def comparativo_edad_primeraysegunda():
    piezas = []
    escenarios = {
        'primera_vuelta': 'primera_vuelta',
        'cepeda_vs_abelardo': 'sv_cepeda_vs_espriella',
        'cepeda_vs_paloma': 'sv_cepeda_vs_valencia',
    }
    for nombre, columna in escenarios.items():
        tabla = votos_por_edad(columna)
        if tabla.empty:
            continue
        long = tabla.reset_index().melt(id_vars='edad_grupo', var_name='opcion', value_name='pct')
        long['escenario'] = nombre
        piezas.append(long)
    if not piezas:
        return pd.DataFrame()
    return pd.concat(piezas, ignore_index=True)

# -----------------------------------------------------------------------------
# Ejecuciones listas para inspección rápida
# -----------------------------------------------------------------------------

# 1) Sesgo relativo por encuestadora vs promedio de las otras
tabla_sesgo_relativo = tabla_sesgos_relativos()
sesgo_candidato_atlas = sesgo_por_candidato('Atlas Intel')
sesgo_candidato_gad3 = sesgo_por_candidato('GAD3')
sesgo_candidato_invamer = sesgo_por_candidato('Invamer')
sesgo_candidato_cnc = sesgo_por_candidato('CNC')

# 2) Perfil demográfico ponderado de los indecisos
indecisos_edad = perfil_indecisos('edad_grupo')
indecisos_region = perfil_indecisos('region') if 'region' in df.columns else pd.DataFrame()

# 3) Distribución de voto según aprobación a Petro
tabla_aprobacion_petro = votos_segun_aprobacion_petro()

# 4) Edad vs primera vuelta y vs las dos segundas vueltas pedidas
edad_primera_vuelta = votos_por_edad('primera_vuelta')
edad_cepeda_abelardo = votos_por_edad('sv_cepeda_vs_espriella')
edad_cepeda_paloma = votos_por_edad('sv_cepeda_vs_valencia')
comparativo_edades = comparativo_edad_primeraysegunda()
aprobacion_region_ponderada = (
    crosstab_ponderado(filtrar_encuesta(), 'region', 'aprobacion_petro', usar_pesos_encuesta=True)
    if 'region' in df.columns and 'aprobacion_petro' in df.columns else pd.DataFrame()
)

# Vistas rápidas
print('Variables demográficas disponibles:', DEMOGRAFICAS)
print('\nSesgo relativo vs otras encuestadoras (edad_grupo):')
display(tabla_sesgo_relativo[tabla_sesgo_relativo['variable'] == 'edad_grupo'].head(20))
print('\nSesgo por candidato - Atlas Intel / edad_grupo:')
display(sesgo_candidato_atlas.head(20))
print('\nPerfil final de indecisos / edad_grupo:')
display(indecisos_edad.head(10))
print('\nVoto según aprobación a Petro:')
display(tabla_aprobacion_petro.head(12))
print('\nEdad vs primera vuelta:')
display(edad_primera_vuelta.head(12))
print('\nEdad vs segunda vuelta Cepeda vs Abelardo:')
display(edad_cepeda_abelardo)
print('\nEdad vs segunda vuelta Cepeda vs Paloma:')
display(edad_cepeda_paloma)
print('\nAprobación presidencial ponderada por región:')
display(aprobacion_region_ponderada)
print('\nComparativo largo por edad:')
display(comparativo_edades.head(30))

Variables demográficas disponibles: ['edad_grupo', 'region', 'estrato', 'educacion', 'zona', 'departamento', 'municipio']

Sesgo relativo vs otras encuestadoras (edad_grupo):


,encuestadora,variable,categoria,peso_encuestadora,peso_promedio_otras,sesgo_rel_pp
0,Atlas Intel,edad_grupo,55+,42.8,27.6,15.2
1,Atlas Intel,edad_grupo,18-34,37.5,36.9,0.6
2,Atlas Intel,edad_grupo,35-54,19.7,35.4,-15.7
3,CNC,edad_grupo,35-54,35.1,30.2,4.9
4,CNC,edad_grupo,18-34,36.6,37.2,-0.6
5,CNC,edad_grupo,55+,28.3,32.5,-4.2
6,GAD3,edad_grupo,35-54,35.4,30.1,5.3
7,GAD3,edad_grupo,18-34,36.2,37.4,-1.2
8,GAD3,edad_grupo,55+,28.3,32.5,-4.2
9,Invamer,edad_grupo,35-54,35.6,30.1,5.5



Sesgo por candidato - Atlas Intel / edad_grupo:


,primera_vuelta,edad_grupo,ponderado_pct,sin_peso_pct,sesgo_pp
1,Abelardo de la Espriella,25-34,19.2,13.4,5.8
0,Abelardo de la Espriella,18-24,7.0,5.3,1.7
2,Abelardo de la Espriella,35-44,21.3,22.5,-1.2
4,Abelardo de la Espriella,60+,19.7,22.0,-2.3
3,Abelardo de la Espriella,45-59,32.7,36.8,-4.1
5,Aníbal Gaviria,18-24,40.0,9.7,30.3
8,Aníbal Gaviria,45-59,30.9,31.1,-0.2
9,Aníbal Gaviria,60+,13.8,20.4,-6.6
7,Aníbal Gaviria,35-44,9.6,20.4,-10.8
6,Aníbal Gaviria,25-34,5.6,18.4,-12.8



Perfil final de indecisos / edad_grupo:


...

sv_cepeda_vs_valencia,Iván Cepeda,NS/NR,Ninguno,No sé,No votaría,Paloma Valencia,Voto en blanco
edad_grupo,,,,,,,
18-24,61.1,2.5,10.3,0.0,0.0,26.1,0.0
18-34,51.1,3.0,0.0,0.0,5.6,28.9,11.3
25-34,50.5,3.3,10.5,0.0,0.0,35.7,0.0
35-44,47.8,3.8,9.8,0.0,0.0,38.6,0.0
35-54,40.2,2.0,0.0,0.0,6.9,43.5,7.4
45-54,37.6,5.5,10.7,0.0,0.0,46.2,0.0
45-59,32.9,0.0,0.0,3.0,0.0,51.8,12.3
55+,33.7,6.0,8.6,0.0,0.0,51.6,0.0
60+,34.9,0.0,0.0,2.2,0.0,48.3,14.6



Aprobación presidencial ponderada por región:


aprobacion_petro,Aprueba,Desaprueba,Excelente / bueno,Malo / muy malo,NS/NR,Negativa,Positiva,Regular
region,,,,,,,,
Amazonía - Orinoquía,0.0,0.0,37.3,40.7,0.0,0.0,0.0,22.0
Bogotá,0.0,0.0,0.0,0.0,3.0,45.3,51.7,0.0
Caribe,0.0,0.0,0.0,0.0,4.5,28.3,67.1,0.0
Central,0.0,0.0,28.4,55.6,0.0,0.0,0.0,16.0
Centro - Oriente,0.1,0.1,0.0,0.0,4.7,49.4,45.8,0.0
Centro - Sur - Amazonía,0.0,0.0,0.0,0.0,7.9,52.1,40.0,0.0
Eje Cafetero,0.0,0.0,0.0,0.0,6.6,59.3,34.1,0.0
Llano,0.0,0.0,0.0,0.0,6.5,49.7,43.8,0.0
Pacífico,0.0,0.0,0.0,0.0,7.1,31.4,61.4,0.0



Comparativo largo por edad:


,edad_grupo,opcion,pct,escenario
0,18-24,Abelardo de la Espriella,7.5,primera_vuelta
1,18-34,Abelardo de la Espriella,13.3,primera_vuelta
2,25-34,Abelardo de la Espriella,13.7,primera_vuelta
3,35-44,Abelardo de la Espriella,17.1,primera_vuelta
4,35-54,Abelardo de la Espriella,23.9,primera_vuelta
5,45-54,Abelardo de la Espriella,18.5,primera_vuelta
6,45-59,Abelardo de la Espriella,35.9,primera_vuelta
7,55+,Abelardo de la Espriella,18.6,primera_vuelta
8,60+,Abelardo de la Espriella,31.0,primera_vuelta
9,18-24,Aníbal Gaviria,0.0,primera_vuelta


## Tablas finales y exportación

Tablas consolidadas para indecisos por demográfica, aprobación a Petro por región y exportación a Excel con una hoja por tabla.

In [29]:
def tabla_indecisos_por_demografica(col_demo, fecha=None):
    return perfil_indecisos(col_demo=col_demo, fecha=fecha)

def tablas_indecisos(fecha=None):
    tablas = {}
    for col_demo in DEMOGRAFICAS + (['aprobacion_petro'] if 'aprobacion_petro' in df.columns else []):
        tabla = tabla_indecisos_por_demografica(col_demo, fecha=fecha)
        if not tabla.empty:
            tablas[col_demo] = tabla
    return tablas

def aprobacion_por_region(fecha=None, usar_pesos_encuesta=True):
    sub = filtrar_encuesta(fecha=fecha)
    if 'region' not in sub.columns or 'aprobacion_petro' not in sub.columns:
        return pd.DataFrame()
    return crosstab_ponderado(sub, 'region', 'aprobacion_petro', usar_pesos_encuesta=usar_pesos_encuesta)

def exportar_tablas_finales(ruta=r'c:\Users\pablo\Downloads\declaraciones_senado\OneDrive_4_12-5-2026\analisis_final.xlsx'):
    tablas = {
        'sesgo_relativo': tabla_sesgo_relativo,
        'sesgo_candidato_atlas': sesgo_candidato_atlas,
        'indecisos_edad_grupo': tabla_indecisos_por_demografica('edad_grupo'),
        'indecisos_region': tabla_indecisos_por_demografica('region') if 'region' in df.columns else pd.DataFrame(),
        'indecisos_estrato': tabla_indecisos_por_demografica('estrato') if 'estrato' in df.columns else pd.DataFrame(),
        'indecisos_educacion': tabla_indecisos_por_demografica('educacion') if 'educacion' in df.columns else pd.DataFrame(),
        'indecisos_sexo': tabla_indecisos_por_demografica('sexo') if 'sexo' in df.columns else pd.DataFrame(),
        'indecisos_aprobacion_petro': tabla_indecisos_por_demografica('aprobacion_petro') if 'aprobacion_petro' in df.columns else pd.DataFrame(),
        'aprobacion_petro': tabla_aprobacion_petro,
        'aprobacion_por_region': aprobacion_por_region(),
        'edad_primera_vuelta': edad_primera_vuelta,
        'edad_cepeda_abelardo': edad_cepeda_abelardo,
        'edad_cepeda_paloma': edad_cepeda_paloma,
        'comparativo_edades': comparativo_edades,
        'df_long6': ponderar(df_long6, groupby=['escenario', 'primera_vuelta', 'voto_sv']),
    }

    with pd.ExcelWriter(ruta, engine='openpyxl') as writer:
        for nombre, tabla in tablas.items():
            if tabla is None or tabla.empty:
                continue
            if isinstance(tabla, pd.Series):
                tabla = tabla.to_frame(name='valor')
            sheet = nombre[:31]
            tabla.to_excel(writer, sheet_name=sheet)

    return tablas

# Resumen rápido para revisar en pantalla
tablas_indecisos = tablas_indecisos()
aprobacion_region = aprobacion_por_region()
print('Tablas de indecisos generadas:', list(tablas_indecisos.keys()))
display(tablas_indecisos.get('edad_grupo', pd.DataFrame()).head(10))
print('\nAprobación a Petro por región:')
display(aprobacion_region)

# Exporta el Excel final con una hoja por tabla
tablas_exportadas = exportar_tablas_finales()
print(r'Excel generado: c:\Users\pablo\Downloads\declaraciones_senado\OneDrive_4_12-5-2026\analisis_final.xlsx')

Tablas de indecisos generadas: ['edad_grupo', 'region', 'estrato', 'educacion', 'zona', 'departamento', 'municipio', 'aprobacion_petro']


,pct_final
edad_grupo,
18-34,47.70
35-44,34.11
35-54,33.70
18-24,21.38
25-34,20.50
60+,14.83
55+,13.26
45-54,10.56
45-59,9.94



Aprobación a Petro por región:


aprobacion_petro,Aprueba,Desaprueba,Excelente / bueno,Malo / muy malo,NS/NR,Negativa,Positiva,Regular
region,,,,,,,,
Amazonía - Orinoquía,0.0,0.0,37.3,40.7,0.0,0.0,0.0,22.0
Bogotá,0.0,0.0,0.0,0.0,3.0,45.3,51.7,0.0
Caribe,0.0,0.0,0.0,0.0,4.5,28.3,67.1,0.0
Central,0.0,0.0,28.4,55.6,0.0,0.0,0.0,16.0
Centro - Oriente,0.1,0.1,0.0,0.0,4.7,49.4,45.8,0.0
Centro - Sur - Amazonía,0.0,0.0,0.0,0.0,7.9,52.1,40.0,0.0
Eje Cafetero,0.0,0.0,0.0,0.0,6.6,59.3,34.1,0.0
Llano,0.0,0.0,0.0,0.0,6.5,49.7,43.8,0.0
Pacífico,0.0,0.0,0.0,0.0,7.1,31.4,61.4,0.0


Tablas de indecisos generadas: ['edad_grupo', 'region', 'estrato', 'educacion', 'zona', 'departamento', 'municipio', 'aprobacion_petro']


,pct_final
edad_grupo,
18-34,47.70
35-44,34.11
35-54,33.70
18-24,21.38
25-34,20.50
60+,14.83
55+,13.26
45-54,10.56
45-59,9.94



Aprobación a Petro por región:


aprobacion_petro,Aprueba,Desaprueba,Excelente / bueno,Malo / muy malo,NS/NR,Negativa,Positiva,Regular
region,,,,,,,,
Amazonía - Orinoquía,0.0,0.0,37.3,40.7,0.0,0.0,0.0,22.0
Bogotá,0.0,0.0,0.0,0.0,3.0,45.3,51.7,0.0
Caribe,0.0,0.0,0.0,0.0,4.5,28.3,67.1,0.0
Central,0.0,0.0,28.4,55.6,0.0,0.0,0.0,16.0
Centro - Oriente,0.1,0.1,0.0,0.0,4.7,49.4,45.8,0.0
Centro - Sur - Amazonía,0.0,0.0,0.0,0.0,7.9,52.1,40.0,0.0
Eje Cafetero,0.0,0.0,0.0,0.0,6.6,59.3,34.1,0.0
Llano,0.0,0.0,0.0,0.0,6.5,49.7,43.8,0.0
Pacífico,0.0,0.0,0.0,0.0,7.1,31.4,61.4,0.0


Excel generado: c:\Users\pablo\Downloads\declaraciones_senado\OneDrive_4_12-5-2026\analisis_final.xlsx
